[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/05_Delay_and_Stability.ipynb)

# DiveLab

## Notebook 05 — Delay, Oscillation and Loss of Stability

**Guiding question:** What happens when corrective action is based on an old state of the system?

*Feedback can stabilize an unstable plant — but delay can weaken or even reverse that benefit.*

## Learning objectives

By the end of this lab, you will be able to:

- explain why delay is important in feedback control;
- distinguish sensing, decision, actuation and plant delay;
- simulate delayed state feedback;
- compare zero-delay and delayed closed-loop responses;
- identify overshoot, oscillation and destabilization caused by delay;
- interpret delay as phase lag;
- connect human buoyancy control with delayed feedback systems.

## From Notebook 04 to Notebook 05

Notebook 04 introduced a controller that changes gas volume in response to state error.

Without delay:

$$
u(t)=\phi(x(t))
$$

The controller reacts to the **current** state.

A real diver cannot do this instantaneously.

Instead, the action may depend on an earlier state:

$$
u(t)=\phi(x(t-\tau))
$$

where:

$$
\tau>0
$$

is the delay.

## Where does delay come from?

In a human-controlled buoyancy loop, several delays can appear:

1. **Sensing delay**  
   The diver first notices a depth or velocity error.

2. **Decision delay**  
   The diver decides whether to inflate, vent, change breathing or wait.

3. **Actuation delay**  
   The button or valve action does not instantaneously change buoyancy.

4. **Plant-response delay**  
   Gas-volume change affects buoyancy, then acceleration, then velocity, then depth.

So even a skilled controller operates with finite response time.

## Why delay is dangerous in an unstable plant

Suppose the diver is already moving upward.

A delayed controller may still be reacting to an earlier instant when the diver was lower or moving more slowly.

It may therefore add too little corrective action — or even the wrong amount.

By the time the correction takes effect, the state may already have changed significantly.

This can produce:

- overshoot;
- repeated overcorrection;
- oscillation;
- loss of stability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Physical model

We use the same simplified plant as Notebook 04.

States:

$$
x=
\begin{bmatrix}
z\\
v\\
V_s
\end{bmatrix}
$$

where:

- $z$ is depth;
- $v$ is upward velocity;
- $V_s$ is equivalent surface gas volume.

In [ ]:
rho = 1025.0
g = 9.80665
P0 = 101325.0

mass = 90.0
z_e = 20.0
v_e = 0.0

gas_surface_volume_e = 0.005

Cd = 0.9
A_drag = 0.7

In [ ]:
def pressure_at_depth(z):
    return P0 + rho * g * z


def gas_volume_at_depth(z, surface_volume):
    return surface_volume * P0 / pressure_at_depth(z)

In [ ]:
gas_volume_e = gas_volume_at_depth(z_e, gas_surface_volume_e)
fixed_volume = mass / rho - gas_volume_e

def buoyant_force(z, surface_gas_volume):
    Vg = gas_volume_at_depth(z, surface_gas_volume)
    return rho * g * (fixed_volume + Vg)

def drag_force(v):
    return 0.5 * rho * Cd * A_drag * v * abs(v)

def acceleration(z, v, surface_gas_volume):
    return (
        buoyant_force(z, surface_gas_volume)
        - mass * g
        - drag_force(v)
    ) / mass

## Baseline controller

We start with the same structure used previously:

$$
u=K_z(z-z_e)-K_vv
$$

For now, the controller sees the current state.

In [ ]:
Kz = 0.00008
Kv = 0.0008

def controller(z, v):
    e_z = z - z_e
    return Kz * e_z - Kv * v

## Simulation without delay

In [ ]:
def simulate_no_delay(
    z0=z_e,
    v0=0.05,
    Vs0=gas_surface_volume_e,
    duration=40.0,
    dt=0.01,
    u_limit=0.0005
):
    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    z = np.zeros(n)
    v = np.zeros(n)
    Vs = np.zeros(n)
    u_hist = np.zeros(n)

    z[0] = z0
    v[0] = v0
    Vs[0] = Vs0

    for i in range(n - 1):
        u = controller(z[i], v[i])
        u = np.clip(u, -u_limit, u_limit)

        a = acceleration(z[i], v[i], Vs[i])

        v[i + 1] = v[i] + a * dt
        z[i + 1] = z[i] - v[i + 1] * dt
        Vs[i + 1] = max(Vs[i] + u * dt, 0.0)

        u_hist[i] = u

    u_hist[-1] = u_hist[-2]
    return t, z, v, Vs, u_hist

In [ ]:
t0, z0, v0, Vs0, u0 = simulate_no_delay()

In [ ]:
plt.plot(t0, z0)
plt.axhline(z_e, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Closed-loop response without delay")
plt.grid(True)
plt.show()

## Add a pure delay

Now let the controller act on:

$$
z(t-\tau)
$$

and:

$$
v(t-\tau)
$$

instead of the current state.

Numerically, if the timestep is $\Delta t$, then the delay corresponds approximately to:

$$
N_\tau=\frac{\tau}{\Delta t}
$$

stored samples.

In [ ]:
def simulate_with_delay(
    delay_s,
    z0=z_e,
    v0=0.05,
    Vs0=gas_surface_volume_e,
    duration=40.0,
    dt=0.01,
    u_limit=0.0005
):
    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    z = np.zeros(n)
    v = np.zeros(n)
    Vs = np.zeros(n)
    u_hist = np.zeros(n)

    z[0] = z0
    v[0] = v0
    Vs[0] = Vs0

    delay_steps = int(round(delay_s / dt))

    for i in range(n - 1):
        j = max(0, i - delay_steps)

        z_delayed = z[j]
        v_delayed = v[j]

        u = controller(z_delayed, v_delayed)
        u = np.clip(u, -u_limit, u_limit)

        a = acceleration(z[i], v[i], Vs[i])

        v[i + 1] = v[i] + a * dt
        z[i + 1] = z[i] - v[i + 1] * dt
        Vs[i + 1] = max(Vs[i] + u * dt, 0.0)

        u_hist[i] = u

    u_hist[-1] = u_hist[-2]
    return t, z, v, Vs, u_hist

## Compare different delays

We test:

- $\tau=0$ s;
- $\tau=0.5$ s;
- $\tau=1.0$ s;
- $\tau=2.0$ s.

In [ ]:
delays = [0.0, 0.5, 1.0, 2.0]
results = {}

for tau in delays:
    results[tau] = simulate_with_delay(tau)

In [ ]:
for tau in delays:
    t, z, v, Vs, u = results[tau]
    plt.plot(t, z, label=f"delay = {tau:.1f} s")

plt.axhline(z_e, linestyle="--", label="Target depth")
plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Effect of feedback delay on depth response")
plt.grid(True)
plt.legend()
plt.show()

## What to look for

As delay increases, look for:

- larger overshoot;
- slower recovery;
- oscillations;
- increasing oscillation amplitude;
- possible divergence from the target.

Delay does not merely make the controller slower.

It changes the dynamics of the closed loop.

In [ ]:
for tau in delays:
    t, z, v, Vs, u = results[tau]
    plt.plot(t, v, label=f"delay = {tau:.1f} s")

plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Upward velocity [m/s]")
plt.title("Effect of feedback delay on vertical velocity")
plt.grid(True)
plt.legend()
plt.show()

## Delayed control action

The control signal itself is based on past information.

Let's inspect it.

In [ ]:
for tau in delays:
    t, z, v, Vs, u = results[tau]
    plt.plot(t, u, label=f"delay = {tau:.1f} s")

plt.xlabel("Time [s]")
plt.ylabel("Control input u [m³/s]")
plt.title("Control effort with different delays")
plt.grid(True)
plt.legend()
plt.show()

## Why oscillation appears

Imagine the diver has started ascending.

A zero-delay controller detects:

$$
v>0
$$

and immediately tends to vent gas.

A delayed controller may still be reacting to an earlier state where:

$$
v \approx 0
$$

or where the depth error had the opposite tendency.

When the correction finally arrives, the diver may already have crossed the desired state.

The controller then reacts again — but once more too late.

This can create repeated overcorrection.

## Delay as phase lag

In classical control theory, a pure time delay is represented in the Laplace domain by:

$$
e^{-s\tau}
$$

This factor has magnitude:

$$
|e^{-j\omega\tau}|=1
$$

but phase:

$$
\angle e^{-j\omega\tau}=-\omega\tau
$$

So delay contributes **phase lag**.

The higher the frequency, the larger the phase lag.

## Why phase lag threatens stability

Negative feedback works because the corrective action points against the error.

But enough phase lag can make the feedback arrive so late that the correction is no longer aligned with the current error.

In the extreme case, negative feedback can behave partly like positive feedback.

This is one reason delay reduces stability margins.

## Human interpretation

For a diver, this means:

> do not wait for a large depth error before reacting.

A delayed, large correction can cause overshoot.

Small, early corrections tend to be easier to manage because the system state has not yet moved far while the response is developing.

This notebook is a control-systems model, not diving-procedure guidance, but the physical intuition is useful.

## Phase-plane comparison

Because delay makes the system depend on its history, the true delayed system is no longer fully described by only $(z,v,V_s)$.

Strictly speaking, delay makes the state space effectively higher-dimensional.

Still, projecting the trajectories onto the $(z,v)$ plane is useful for visualization.

In [ ]:
for tau in delays:
    t, z, v, Vs, u = results[tau]
    plt.plot(z, v, label=f"delay = {tau:.1f} s")

plt.scatter([z_e], [0], s=70, label="Equilibrium")
plt.xlabel("Depth z [m]")
plt.ylabel("Upward velocity v [m/s]")
plt.title("Projected phase trajectories with delay")
plt.grid(True)
plt.legend()
plt.show()

## A systems-theory warning

For ordinary differential equations:

$$
\dot x=f(x)
$$

the future is determined by the current state $x(t)$.

For a delay differential equation:

$$
\dot x(t)=f(x(t),x(t-\tau))
$$

the future depends on a segment of past history.

So the mathematical object is no longer a finite-dimensional ODE in the usual sense.

This is one reason delay systems are mathematically richer.

## Delay differential equation viewpoint

A simplified delayed feedback model can be written as:

$$
\dot x(t)
=
Ax(t)+Bu(t)
$$

with:

$$
u(t)=-Kx(t-\tau)
$$

therefore:

$$
\dot x(t)
=
Ax(t)-BKx(t-\tau)
$$

This is a delay differential equation.

## Characteristic equation

For a delayed linear system, the characteristic equation contains an exponential term.

Schematically:

$$
\det\left(
sI-A+BK e^{-s\tau}
\right)=0
$$

Unlike an ordinary polynomial characteristic equation, this generally has infinitely many roots.

That is another major difference between ODE control systems and delayed systems.

## Delay margin intuition

For fixed controller gains, there may be a maximum delay:

$$
\tau_{\max}
$$

below which the closed loop remains acceptably stable and above which oscillation or divergence appears.

This is related to the concept of **delay margin**.

In practice, the allowable delay depends on:

- plant dynamics;
- controller gains;
- actuator dynamics;
- saturation;
- nonlinearities.

## A numerical delay sweep

Let's measure how the maximum depth deviation changes as delay increases.

In [ ]:
delay_grid = np.linspace(0.0, 3.0, 31)
max_depth_error = []

for tau in delay_grid:
    t, z, v, Vs, u = simulate_with_delay(tau)
    max_depth_error.append(np.max(np.abs(z - z_e)))

plt.plot(delay_grid, max_depth_error)
plt.xlabel("Delay [s]")
plt.ylabel("Maximum |depth error| [m]")
plt.title("Sensitivity of closed-loop performance to delay")
plt.grid(True)
plt.show()

## Interpreting the delay sweep

A rising curve means that performance degrades as delay increases.

If the error begins to grow dramatically, that suggests the controller is approaching a region where the delayed closed-loop dynamics are poorly damped or unstable.

This numerical experiment is a first approximation to a delay-margin study.

## Gain and delay interact

A stronger controller is not always better.

Large gains can improve response when delay is negligible, but they may also make the system more sensitive to delay.

This creates a classic control tradeoff:

> faster correction vs robustness to delay.

## Experiment: stronger feedback

In [ ]:
Kz_original = Kz
Kv_original = Kv

Kz = 0.00014
Kv = 0.0014

strong_results = {}
for tau in delays:
    strong_results[tau] = simulate_with_delay(tau)

for tau in delays:
    t, z, v, Vs, u = strong_results[tau]
    plt.plot(t, z, label=f"delay = {tau:.1f} s")

plt.axhline(z_e, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Stronger feedback: delay becomes more important")
plt.grid(True)
plt.legend()
plt.show()

# Restore baseline gains
Kz = Kz_original
Kv = Kv_original

## Robustness

A controller is not judged only by how well it performs under ideal conditions.

We also care about how well it tolerates:

- delay;
- modeling error;
- disturbances;
- actuator limits;
- uncertain parameters.

This is the beginning of the concept of **robust control**.

## Exercises

### 1. Find a problematic delay

Increase `delay_s` gradually.

At what delay do you first see clearly sustained or growing oscillation?

Record the value for your chosen gains.

### 2. Change the gains

Try increasing and decreasing:

```python
Kz
Kv
```

How does the sensitivity to delay change?

### 3. Velocity feedback only

Set:

```python
Kz = 0
```

and investigate different delays.

What changes?

### 4. Depth feedback only

Set:

```python
Kv = 0
```

Repeat the experiment.

Compare the behavior with velocity feedback.

### 5. Change the initial disturbance

Try:

```python
v0 = 0.01
v0 = 0.10
```

Does the apparent effect of delay change because of nonlinearities and saturation?

## Challenge — estimate a numerical delay margin

Define a practical criterion such as:

- maximum depth error below a chosen threshold;
- no sustained growth during 40 s;
- velocity remains within a selected bound.

Then sweep over delay values and estimate the largest acceptable delay.

This is not a formal analytical delay margin, but it is a useful simulation-based engineering estimate.

In [ ]:
# Your code here

## Summary

In this lab we learned that:

- real feedback is rarely instantaneous;
- delayed feedback uses past state information;
- delay can increase overshoot and oscillation;
- sufficient delay can threaten closed-loop stability;
- in the Laplace domain, delay contributes the factor $e^{-s\tau}$;
- delay introduces frequency-dependent phase lag;
- delayed systems are naturally described by delay differential equations;
- stronger feedback can reduce robustness to delay;
- controller design involves a tradeoff between speed and stability margin.

### Core insight

> **A correct action applied too late can become a poor correction.**

### Next

Notebook 06 can introduce **sensor noise and estimation**:

> What happens when the controller sees the state quickly, but inaccurately?